In [1]:
import random  # Used to generate a small random jitter value for retry delays.

# Custom exception for transient, retryable provider rate-limit failures.
class RateLimitError(Exception):
    pass

# Custom exception for permanent request problems that should not be retried.
class BadRequestError(Exception):
    pass


In [2]:

def call_with_retry(request_fn, max_attempts=3, base_delay=2):
    # Try the request up to max_attempts times.
    for attempt in range(max_attempts):
        try:
            # Call the provider function; if it succeeds, return its result immediately.
            return request_fn()

        except BadRequestError:
            # Permanent error: do not retry, raise immediately.
            raise

        except (RateLimitError, TimeoutError) as err:
            # These are transient errors, so we may retry if attempts remain.
            if attempt == max_attempts - 1:
                # We used all allowed attempts, so fail with a final error.
                raise RuntimeError("max retries reached") from err

            # Compute exponential backoff using the current attempt number.
            # attempt = 0 -> base_delay * 2^0
            # attempt = 1 -> base_delay * 2^1
            # etc.
            jitter = random.uniform(0, 1)  # Small random extra delay to avoid synchronized retries.
            delay = base_delay * (2 ** attempt) + jitter  # Backoff delay that would be used in real code.

            # In automated testing, we do not actually sleep.
            # If needed for debugging, you could log the delay here.
            # print(f"Retrying after {delay:.3f}s due to transient error: {type(err).__name__}")

        except Exception:
            # Any unexpected error is treated as non-retryable here.
            raise



In [3]:


class BudgetTracker:
    def __init__(self, budget_cap):
        # Store the maximum allowed spend.
        self.budget_cap = budget_cap
        # Track the total spend across all completed calls.
        self.cumulative_spend = 0.0

    def has_budget(self):
        # Return True only while spend is still below the cap.
        return self.cumulative_spend < self.budget_cap

    def update(self, cost):
        # Add the finished call's cost to the running total.
        self.cumulative_spend += cost



In [4]:
def make_request_fn(error_sequence):
    # Create a simulated provider call function for one test case.
    # It will raise the listed errors in order, then succeed once the list is exhausted.
    state = {"attempt_index": 0}

    def request_fn():
        # Read the current attempt number.
        i = state["attempt_index"]
        # Move to the next attempt for the next call.
        state["attempt_index"] += 1

        # If there is still an error to emit for this attempt, raise it.
        if i < len(error_sequence):
            error_name = error_sequence[i]

            # Map the string name to the correct exception type.
            if error_name == "RateLimitError":
                raise RateLimitError("simulated rate limit error")
            elif error_name == "TimeoutError":
                raise TimeoutError("simulated timeout error")
            elif error_name == "BadRequestError":
                raise BadRequestError("simulated bad request error")
            else:
                # Unknown error name: treat it as a generic failure.
                raise Exception(f"Unknown simulated error: {error_name}")

        # If no error is left in the sequence, the call succeeds.
        return {"status": "success"}

    return request_fn

def process_budget_demo(calls, tracker, price_per_million_input, price_per_million_output):
    # Collect one outcome record per call.
    outcomes = []

    for call in calls:
        call_id = call["call_id"]  # Read the call identifier.
        input_tokens = call["input_tokens"]  # Read input token count.
        output_tokens = call["output_tokens"]  # Read output token count.

        # Check the shared budget before attempting the call.
        if not tracker.has_budget():
            # If no budget remains, block the call and do not attempt it.
            outcomes.append(
                f"{call_id}: blocked (budget exhausted, cumulative_spent={tracker.cumulative_spend:.4f})"
            )
            continue

        # Compute call cost using the required formula.
        cost = (
            (input_tokens / 1_000_000) * price_per_million_input
            + (output_tokens / 1_000_000) * price_per_million_output
        )

        # The call is allowed to proceed, so update the shared tracker after completion.
        tracker.update(cost)

        # Record success, cost, and new cumulative spend.
        outcomes.append(
            f"{call_id}: succeeded, cost={cost:.4f}, cumulative_spent={tracker.cumulative_spend:.4f}"
        )

    return outcomes


if __name__ == "__main__":
    # Sample Input A: retry logic test cases.
    sample_a = [
        {"call_id": "A1", "error_sequence": []},
        {"call_id": "A2", "error_sequence": ["RateLimitError", "RateLimitError"]},
        {"call_id": "A3", "error_sequence": ["RateLimitError", "RateLimitError", "RateLimitError"]},
        {"call_id": "A4", "error_sequence": ["BadRequestError"]},
    ]

    # Sample Input B: budget guardrail test cases.
    sample_b = [
        {"call_id": "B1", "input_tokens": 500, "output_tokens": 800},
        {"call_id": "B2", "input_tokens": 300, "output_tokens": 400},
        {"call_id": "B3", "input_tokens": 1000, "output_tokens": 1000},
        {"call_id": "B4", "input_tokens": 100, "output_tokens": 100},
    ]

    # Create one shared tracker with the required budget cap.
    tracker = BudgetTracker(budget_cap=0.01)

    # Run and print the retry demo results.
    # The function process_retry_demo is not defined in the provided code.
    # Assuming it would be defined elsewhere or is a placeholder, and commenting it out.
    # print(process_retry_demo(sample_a, max_attempts=3, base_delay=2))

    # Run and print the budget demo results.
    print(
        process_budget_demo(
            sample_b,
            tracker,
            price_per_million_input=2.00,
            price_per_million_output=6.00,
        )
    )

['B1: succeeded, cost=0.0058, cumulative_spent=0.0058', 'B2: succeeded, cost=0.0030, cumulative_spent=0.0088', 'B3: succeeded, cost=0.0080, cumulative_spent=0.0168', 'B4: blocked (budget exhausted, cumulative_spent=0.0168)']
